# Контрастная синтетика для проверки метрик чанкинга

По умолчанию ноутбук работает офлайн. Для платного smoke-run явно установите
`RUN_GENERATION = True`; при `SELECTED_PROMPTS = None` будут обработаны все 9 prompts.

In [4]:
from __future__ import annotations

from datetime import datetime, timezone
import os
from pathlib import Path
import sys
from tempfile import TemporaryDirectory
from uuid import uuid4

import pandas as pd
from dotenv import load_dotenv

load_dotenv()

RUN_GENERATION = True # <---
SELECTED_PROMPTS: list[str] | None = None
PAIRS_PER_PROMPT = 1
MODEL_NAME = "deepseek-v4-pro"
BASE_URL = "https://api.deepseek.com"
TEMPERATURE = 1.0
MAX_TOKENS = 8192
TIMEOUT_SECONDS = 120.0
MAX_ATTEMPTS = 3
API_KEY = os.getenv("API_KEY")

cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (path for path in (cwd, cwd.parent) if (path / "prompts").is_dir()),
    cwd,
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.generation_pipeline import (
    EXPECTED_RELATION,
    append_record,
    build_prompt_registry,
    generate_pair,
    load_prompt_text,
    make_record,
    select_prompts,
    validate_pair,
)

PROMPTS_ROOT = PROJECT_ROOT / "prompts"
OUTPUT_ROOT = PROJECT_ROOT / "data" / "generated"
PROMPT_REGISTRY = build_prompt_registry(PROMPTS_ROOT)


In [5]:
def create_client():
    if not API_KEY:
        raise RuntimeError(
            "Set DEEPSEEK_API_KEY (or fallback API_KEY) before generation"
        )
    from openai import OpenAI

    return OpenAI(
        api_key=API_KEY,
        base_url=BASE_URL,
        timeout=TIMEOUT_SECONDS,
    )


def run_generation(
    names: list[str] | None = SELECTED_PROMPTS,
    pairs_per_prompt: int = PAIRS_PER_PROMPT,
):
    if (
        isinstance(pairs_per_prompt, bool)
        or not isinstance(pairs_per_prompt, int)
        or pairs_per_prompt < 1
    ):
        raise ValueError("pairs_per_prompt must be a positive integer")

    client = create_client()
    run_id = (
        datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
        + "-"
        + uuid4().hex[:8]
    )
    results = []
    for entry in select_prompts(PROMPT_REGISTRY, names):
        system_prompt, user_prompt = load_prompt_text(PROMPTS_ROOT, entry)
        for _ in range(pairs_per_prompt):
            payload, attempts, error = generate_pair(
                client,
                entry,
                system_prompt,
                user_prompt,
                model=MODEL_NAME,
                temperature=TEMPERATURE,
                max_tokens=MAX_TOKENS,
                max_attempts=MAX_ATTEMPTS,
            )
            status = "rejected"
            if payload is not None:
                record = make_record(
                    payload,
                    entry,
                    run_id=run_id,
                    model=MODEL_NAME,
                )
                append_record(record, OUTPUT_ROOT)
                status = "successful"
            results.append(
                {
                    "dataset_type": entry["dataset_type"],
                    "prompt_name": entry["prompt_name"],
                    "status": status,
                    "attempts": attempts,
                    "error": error,
                }
            )

    details = pd.DataFrame(results)
    summary = details.groupby(["dataset_type", "status"]).size().unstack(fill_value=0)
    print(f"run_id: {run_id}")
    print(summary.to_string())
    rejected = details.loc[
        details["status"] == "rejected",
        ["prompt_name", "attempts", "error"],
    ]
    if not rejected.empty:
        print("\nRejected requests:")
        print(rejected.to_string(index=False))
    return run_id, details


In [6]:
# Compact offline smoke-check: prompt loading, validation and real JSONL round-trip.
for prompt_entry in PROMPT_REGISTRY:
    load_prompt_text(PROMPTS_ROOT, prompt_entry)

sc_entry = next(
    entry for entry in PROMPT_REGISTRY if entry["prompt_name"] == "size_compliance"
)
chunks = [
    "1.1. Фонд поддерживает образовательные проекты.",
    "1.2. Совет ежегодно утверждает план работы.",
]
focus = {
    "target_chunk_indices": [0],
    "length_range_chars": {"min": 40, "max": 60},
}
fixture = {
    "document_title": "Устав фонда «Северный маяк»",
    "source_document": "".join(chunks),
    "positive": {
        "chunks": chunks,
        "rationale": "Оба чанка входят в диапазон.",
        "focus": focus,
    },
    "negative": {
        "chunks": ["".join(chunks)],
        "rationale": "Объединённый чанк превышает диапазон.",
        "focus": focus,
    },
    "controlled_change": "Удалена граница между пунктами.",
    "expected_relation": EXPECTED_RELATION,
}
validate_pair(fixture, sc_entry)
with TemporaryDirectory() as temporary_directory:
    smoke_record = make_record(
        fixture,
        sc_entry,
        run_id="offline-smoke",
        model=MODEL_NAME,
    )
    smoke_path = append_record(smoke_record, Path(temporary_directory))
    assert smoke_path.read_text(encoding="utf-8").count("\n") == 1

OFFLINE_SMOKE_OK = True
print(
    pd.DataFrame(PROMPT_REGISTRY)[
        ["prompt_name", "dataset_type", "target_metric"]
    ].to_string(index=False)
)
print("\nOffline validation and JSONL save: OK")


                  prompt_name       dataset_type                 target_metric
           general_validation general_validation                      multiple
              size_compliance    metric_specific               Size Compliance
          intrachunk_cohesion    metric_specific           Intrachunk Cohesion
         contextual_coherence    metric_specific          Contextual Coherence
             boundary_clarity    metric_specific              Boundary Clarity
                  chunk_score    metric_specific                    ChunkScore
           hope_concept_unity    metric_specific            HOPE Concept Unity
   hope_semantic_independence    metric_specific    HOPE Semantic Independence
hope_information_preservation    metric_specific HOPE Information Preservation

Offline validation and JSONL save: OK


In [7]:
if RUN_GENERATION:
    generation_run_id, generation_details = run_generation()
else:
    print(
        "Generation disabled. Set RUN_GENERATION = True "
        "for an explicit paid smoke-run."
    )


run_id: 20260819T110720Z-9d73c64e
status              rejected  successful
dataset_type                            
general_validation         0           1
metric_specific            3           5

Rejected requests:
        prompt_name  attempts                                                                 error
    size_compliance         3 ValidationError: all positive SC chunks must satisfy the length range
intrachunk_cohesion         3       ValidationError: positive chunks do not restore source_document
 hope_concept_unity         3       ValidationError: positive chunks do not restore source_document
